# Belief Representations in POMDPPlanners

This notebook explores the **belief state** representations available in POMDPPlanners. A belief state encodes the agent's uncertainty about the true state of the environment and is the central object that planners reason about.

**Belief Types Covered:**
- **WeightedParticleBelief** — the default belief for most planners, using weighted particles
- **GaussianBelief** — parametric belief using a multivariate normal distribution
- **GaussianMixtureBelief** — mixture of Gaussians for multi-modal posterior distributions

**Key Concepts Demonstrated:**
- Creating and updating particle beliefs manually
- The effect of resampling on Effective Sample Size (ESS)
- Kalman filter variants (Linear, Extended, Unscented)
- Gaussian mixture beliefs with custom updaters

## 1. Particle Beliefs: The Default

`WeightedParticleBelief` is the most widely used belief representation. It approximates the posterior using a set of weighted particles. The `get_initial_belief()` utility creates one from any environment's initial state distribution.

Let's create a particle belief for the Tiger POMDP and watch the weights concentrate as the agent listens repeatedly.

In [ ]:
import numpy as np
np.random.seed(42)

from POMDPPlanners.environments.tiger_pomdp import TigerPOMDP
from POMDPPlanners.core.belief import get_initial_belief

env = TigerPOMDP(discount_factor=0.95)
belief = get_initial_belief(env, n_particles=20, resampling=False)

print(f"Particles: {belief.particles}")
print(f"Initial weights: {np.round(belief.normalized_weights, 3)}")
print(f"Number of particles: {len(belief.particles)}")
print()

# Pick a true state and listen 5 times
true_state = "tiger_left"
action = "listen"

for step in range(5):
    # Generate an observation from the true state
    observation = env.sample_observation(next_state=true_state, action=action)
    belief = belief.update(action=action, observation=observation, pomdp=env)
    
    # Show the weight on each unique state
    dist = belief.to_unique_support_distribution()
    print(f"Step {step + 1}: obs={observation}")
    for val, prob in zip(dist.values, dist.probs):
        print(f"  P({val}) = {prob:.4f}")

## 2. The Effect of Resampling

Without resampling, particle weights can become highly skewed — a few particles carry almost all the weight while the rest become negligible. **Resampling** redistributes weight by duplicating high-weight particles and discarding low-weight ones.

We measure this using the **Effective Sample Size (ESS)**:

$$\text{ESS} = \frac{1}{\sum_i w_i^2}$$

A higher ESS means more particles are contributing meaningfully to the belief.

In [ ]:
np.random.seed(42)

belief_no_resample = get_initial_belief(env, n_particles=20, resampling=False)
belief_resample = get_initial_belief(env, n_particles=20, resampling=True)

true_state = "tiger_left"
action = "listen"

# Generate a fixed observation sequence
np.random.seed(99)
observations = [
    env.sample_observation(next_state=true_state, action=action)
    for _ in range(8)
]

print(f"{'Step':<6} {'Obs':<14} {'ESS (no resample)':<22} {'ESS (resample)'}")
print("-" * 60)

for step, obs in enumerate(observations):
    belief_no_resample = belief_no_resample.update(action=action, observation=obs, pomdp=env)
    belief_resample = belief_resample.update(action=action, observation=obs, pomdp=env)
    
    ess_no = 1.0 / np.sum(belief_no_resample.normalized_weights ** 2)
    ess_yes = 1.0 / np.sum(belief_resample.normalized_weights ** 2)
    
    print(f"{step + 1:<6} {obs:<14} {ess_no:<22.2f} {ess_yes:.2f}")

print()
print("Resampled belief maintains higher ESS, keeping more particles effective.")

## 3. Gaussian Beliefs: When Structure is Known

When the environment has linear-Gaussian dynamics, a `GaussianBelief` with a Kalman filter updater provides an **exact** Bayesian update instead of an approximation.

The **Continuous Light-Dark POMDP** has linear dynamics, so a Linear Kalman Filter (LKF) is the natural choice. The factory function `create_continuous_light_dark_gaussian_belief()` sets up the correct matrices automatically.

In [3]:
import numpy as np
np.random.seed(42)

from POMDPPlanners.environments.light_dark_pomdp.continuous_light_dark_pomdp import (
    ContinuousLightDarkPOMDP,
)
from POMDPPlanners.environments.light_dark_pomdp.light_dark_pomdp_beliefs.continuous_light_dark_gaussian_beliefs import (
    create_continuous_light_dark_gaussian_belief,
    GaussianBeliefUpdaterType,
)

ld_env = ContinuousLightDarkPOMDP(discount_factor=0.95)
gaussian_belief = create_continuous_light_dark_gaussian_belief(
    env=ld_env,
    updater_type=GaussianBeliefUpdaterType.LINEAR_KALMAN,
    initial_covariance=np.eye(2) * 5.0,
)

print(f"Initial mean: {gaussian_belief.mean}")
print(f"Initial covariance trace: {np.trace(gaussian_belief.covariance):.4f}")
print(f"Initial entropy: {gaussian_belief.entropy():.4f}")
print()

state = ld_env.start_state.astype(float).copy()
action = np.array([0.5, 0.0])

for step in range(5):
    next_state, observation, reward = ld_env.sample_next_step(state, action)
    gaussian_belief = gaussian_belief.update(
        action=action, observation=observation, pomdp=ld_env
    )
    state = next_state
    
    print(
        f"Step {step + 1}: mean={np.round(gaussian_belief.mean, 3)}, "
        f"cov_trace={np.trace(gaussian_belief.covariance):.4f}, "
        f"entropy={gaussian_belief.entropy():.4f}"
    )

print()
print("Covariance trace and entropy decrease as observations provide information.")

## 4. Comparing Kalman Variants

POMDPPlanners provides three Gaussian updater types:
- **Linear Kalman Filter (LKF)** — exact for linear-Gaussian systems
- **Extended Kalman Filter (EKF)** — first-order linearization for nonlinear systems
- **Unscented Kalman Filter (UKF)** — sigma-point propagation for nonlinear systems

Since the Light-Dark POMDP has linear dynamics, all three should produce nearly identical results.

In [4]:
np.random.seed(42)

updater_types = [
    ("LKF", GaussianBeliefUpdaterType.LINEAR_KALMAN),
    ("EKF", GaussianBeliefUpdaterType.EKF),
    ("UKF", GaussianBeliefUpdaterType.UKF),
]

beliefs = {}
for name, updater_type in updater_types:
    beliefs[name] = create_continuous_light_dark_gaussian_belief(
        env=ld_env,
        updater_type=updater_type,
        initial_covariance=np.eye(2) * 5.0,
    )

# Generate a fixed sequence of observations
np.random.seed(42)
state = ld_env.start_state.astype(float).copy()
action = np.array([0.5, 0.0])
obs_sequence = []
states_sequence = [state.copy()]

for _ in range(10):
    next_state, observation, _ = ld_env.sample_next_step(state, action)
    obs_sequence.append(observation)
    state = next_state
    states_sequence.append(state.copy())

# Update all three beliefs with the same observation sequence
for obs in obs_sequence:
    for name in beliefs:
        beliefs[name] = beliefs[name].update(action=action, observation=obs, pomdp=ld_env)

print("Final means after 10 identical updates:")
for name in beliefs:
    print(f"  {name}: {np.round(beliefs[name].mean, 4)}")

# Check maximum difference between any pair
means = [beliefs[name].mean for name in beliefs]
max_diff = max(
    np.max(np.abs(means[i] - means[j]))
    for i in range(len(means)) for j in range(i + 1, len(means))
)
print(f"\nMax difference between any pair: {max_diff:.6f}")
print("All three agree on a linear system (differences are numerical noise).")

## 5. Gaussian Mixture Beliefs

`GaussianMixtureBelief` represents the posterior as a weighted mixture of Gaussians. This is useful when the true posterior is multi-modal — for example, when the agent is uncertain between two distinct hypotheses.

Updates are delegated to a `GaussianMixtureBeliefUpdater` subclass. Below we define a simple one that shifts each component's mean toward the observation.

In [5]:
import numpy as np
np.random.seed(42)

from POMDPPlanners.core.belief import GaussianMixtureBelief
from POMDPPlanners.core.belief.gaussian_mixture_belief import GaussianMixtureBeliefUpdater


class SimpleGMMUpdater(GaussianMixtureBeliefUpdater):
    """Shifts component means toward the observation and shrinks covariances."""

    def __init__(self, learning_rate: float = 0.3, shrink_factor: float = 0.9):
        self._lr = learning_rate
        self._shrink = shrink_factor

    def update(self, means, covariances, weights, action, observation):
        new_means = [
            m + self._lr * (observation - m) for m in means
        ]
        new_covs = [c * self._shrink for c in covariances]
        return new_means, new_covs, weights

    @property
    def config_id(self):
        return f"simple_gmm_lr{self._lr}_shrink{self._shrink}"


# Create a 2-component GMM belief in 2D
means = [np.array([0.0, 0.0]), np.array([5.0, 5.0])]
covs = [np.eye(2) * 2.0, np.eye(2) * 2.0]
weights = np.array([0.5, 0.5])

gmm_belief = GaussianMixtureBelief(
    means=means, covariances=covs, weights=weights,
    updater=SimpleGMMUpdater(learning_rate=0.3, shrink_factor=0.9),
)

print(f"Components: {gmm_belief.n_components}")
print(f"Weights: {gmm_belief.weights}")
print(f"Means: {[m.tolist() for m in gmm_belief.means]}")
print(f"Entropy: {gmm_belief.entropy():.4f}")
print()

# Sample some states
samples = [gmm_belief.sample() for _ in range(5)]
print("Sampled states:")
for s in samples:
    print(f"  {np.round(s, 3)}")
print()

# Update toward observation at [3, 3]
observation = np.array([3.0, 3.0])
updated_gmm = gmm_belief.update(action=0, observation=observation)

print(f"After update toward observation {observation}:")
print(f"  Components: {updated_gmm.n_components}")
print(f"  Means: {[np.round(m, 3).tolist() for m in updated_gmm.means]}")
print(f"  Entropy: {updated_gmm.entropy():.4f}")
print("  Both means shifted toward the observation [3, 3].")

## Summary: When to Use What

| Scenario | Recommended Belief | Why |
|---|---|---|
| General-purpose / first try | `WeightedParticleBelief` via `get_initial_belief()` | Works with all planners, no assumptions needed |
| Linear-Gaussian dynamics | `GaussianBelief` + LKF | Exact Bayesian update, no particle degeneracy |
| Nonlinear dynamics with Gaussian noise | `GaussianBelief` + EKF or UKF | Better approximation than particles for smooth models |
| Multi-modal posterior | `GaussianMixtureBelief` | Captures multiple hypotheses that particles may miss |

## What's Next?

- **Basic usage**: See [basic_usage.ipynb](basic_usage.ipynb) for a full introduction to environments and planners
- **Build your own environment**: See [custom_environment.ipynb](custom_environment.ipynb) to create a POMDP from scratch
- **Inspect planner behavior**: See [tree_analysis_debugging.ipynb](tree_analysis_debugging.ipynb) for MCTS tree diagnostics
- **Compare planners at scale**: See [planners_comparison.ipynb](planners_comparison.ipynb) for batch evaluations
- **Tune hyperparameters**: See [hyperparameter_tuning.ipynb](hyperparameter_tuning.ipynb) for optimizing planner parameters with Optuna